# 1.3 · PD results

*1. Experiment 1 · notebook 1.3 of the story.* ← [1.2 · LGD training](<1.2_lgd_training.ipynb>) · [1.4 · LGD results](<1.4_lgd_results.ipynb>) →

**The benchmark — the experiment's answer.** The 45 Experiment 1 arms and the shared reference
models (the released TabICLv2, TabPFN, CatBoost, a linear model), all scored by the same code on the
same splits and the same context size, read from `output/results/pd/eval/`. The training-time view
of the same arms is [1.1 · PD training](<1.1_pd_training.ipynb>).

**Status: the benchmark has not run yet.** It runs once every arm of the track has trained, so each
figure below is a one-line placeholder until then and fills in when the results exist; the notebook
can be run at any time.

**Two questions, in the protocol's order** (`docs/EXPERIMENTAL_DESIGN.md` §5). **A** — which
configuration does the development split select? It is the only data a prior may be chosen on.
**B** — how does it do on the holdout? Untouched until the end, it is what gets reported. Choosing on
the holdout would invalidate the experiment. **C** then shows every dataset on both sides of the split
and **D** places the numbers beside the literature.

**Reading the metrics.** PD is imbalanced, so accuracy is never reported alone — at a 7 % base rate *never default* already scores 0.93, and default-threshold models collapse to 0 % recall on real credit books (`papers/2026/05_Tanna_DataPresentation`, Table 1). ROC-AUC measures ranking; Brier, log-loss and the calibration slope measure the probabilities a PD model is actually used for. Our nano-scale arms are expected to trail the frontier
reference in absolute terms; the claim is the contrast between priors at matched compute, not state
of the art.

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, pathlib
ROOT = pathlib.Path.cwd()
# Walk up to the repository root — the notebook may be opened from its chapter folder, from
# notebooks/, or from the root — then work FROM the root, so relative paths (config/...) resolve
# exactly as under `python -m src.utils.run_notebooks`.
while not (ROOT / "src" / "visualize").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from src.visualize import results_plots, literature_plots, figures, style, literature

style.apply()
pd.set_option("display.width", 200, "display.max_columns", 40)

TASK = "pd"
EXP = "exp1"
# Reads output/manifests/ and output/results/ — whatever the runs have written so far, so a
# partial sweep still renders. Constructing the saver clears THIS notebook's figure folder only.
FIGS = figures.FigureSaver("1.3_pd_results")

## A · Which configuration does development select?

### A1 · Configurations ranked on the development datasets

**What it shows.** Every configuration's development ROC-AUC, averaged with equal weight per development dataset and then per training seed, sorted, coloured by kind (credit prior, control, external baseline); the whisker is the standard deviation over seeds. A configuration missing a development dataset is left out rather than rewarded for skipping a hard one (`src/eval/selection.py`).

**Why it matters.** The top of this ranking is the prior the next experiment inherits — the only selection the protocol allows.

**What it says.** Not run yet.

In [ ]:
FIGS.save(results_plots.overall_ranking(TASK, exp=EXP), "overall_ranking",
    caption="Development ROC-AUC per configuration, averaged over the development datasets and training seeds, as horizontal bars coloured by model kind with whiskers for the training-seed standard deviation.");

## B · How does it do on the holdout?

The reported result: the same comparison on the holdout datasets, which no selection has seen.

### B1 · Credit prior against the control

**What it shows.** Every model's mean ROC-AUC over the holdout datasets, one point per model, grouped into credit-prior arms, control arms and external baselines, with a bar at each group mean.

**Why it matters.** The figure the experiment exists for: does training on our prior beat training on TabICL's own, at the same compute, on data neither saw?

**What it says.** Not run yet.

In [ ]:
FIGS.save(results_plots.credit_vs_control(TASK, exp=EXP, role="holdout"), "credit_vs_control",
    caption="Per-model mean ROC-AUC on the holdout datasets for credit-prior arms, control arms and external baselines, one point per model with a bar at each group mean.");

### B2 · Against the released reference

**What it shows.** Each trained arm's mean holdout ROC-AUC minus that of the released TabICLv2, one bar per arm, coloured by kind, with the reference at zero.

**Why it matters.** The released model was trained on orders of magnitude more compute; trailing it is expected and not the claim, but how far each arm trails is the scale of the result.

**What it says.** Not run yet.

In [ ]:
FIGS.save(results_plots.beats_reference(TASK, exp=EXP, role="holdout"), "beats_reference",
    caption="Each trained arm's mean ROC-AUC on the holdout datasets minus the released TabICLv2's, one horizontal bar per arm coloured by kind, with a reference line at zero.");

### B3 · Every benchmark metric

**What it shows.** One panel per benchmark metric on the holdout datasets, one bar per kind (credit prior, control, external baseline); each title carries the metric's goal, and a metric with a target value draws it dashed.

**Why it matters.** A prior that helps ranking but hurts calibration splits across these panels.

**What it says.** Not run yet.

In [ ]:
FIGS.save(results_plots.metric_grid(TASK, exp=EXP, role="holdout"), "metric_grid",
    caption="One panel per benchmark metric on the holdout datasets, one bar per model kind (credit prior, control, external baseline); each title marks the improving direction or target value.");

## C · Where does it hold?

No dataset hidden behind a mean: every dataset on both sides of the split, labelled with its side.

### C1 · Every dataset

**What it shows.** The best ROC-AUC of each kind on each real dataset, as grouped bars — development datasets first, then holdout, each labelled with its side of the split.

**Why it matters.** A prior that wins on average by helping one easy dataset is exposed here.

**What it says.** Not run yet.

In [ ]:
FIGS.save(results_plots.per_dataset(TASK, exp=EXP), "per_dataset",
    caption="Best ROC-AUC of each model kind per real dataset as grouped bars, development datasets first and then holdout, each labelled with its side of the split.");

### C2 · The same, as one table

**What it shows.** The best ROC-AUC of each kind on each dataset as an annotated heatmap: datasets down the side (development first), kinds across.

**Why it matters.** The whole pattern at a glance, with the numbers written in.

**What it says.** Not run yet.

In [ ]:
FIGS.save(results_plots.per_dataset_heatmap(TASK, exp=EXP), "per_dataset_heatmap",
    caption="Best ROC-AUC of each model kind on each dataset as an annotated heatmap, development datasets first and then holdout on the vertical axis, model kinds on the horizontal axis.");

## D · Where the field sits

### D1 · Credit-domain ROC-AUC in the literature

**What it shows.** Every ROC-AUC on a real credit dataset that `tfm-library` holds, one bar per published result, sorted: Tanna 2026 on Home Credit and Lending Club (`papers/2026/05_Tanna_DataPresentation` Table 3) and Hollmann 2023 on Credit-g (`papers/2023/09_Hollmann_TabPFN` Table 2); the Home Credit Kaggle ceiling is a value the paper cites, not one it evaluated.

**Why it matters.** Context for our numbers, not a scoreboard: each value comes from its paper's own full-dataset protocol, with tens of thousands of context rows, not the in-context setting used here.

**What it says.** On Home Credit the TFMs reach 0.771 (TabICL) and 0.786 (TabPFN) against a random forest's 0.739; on Credit-g TabPFN reaches 0.789; on Lending Club the best TFM reaches 0.686, below XGBoost's 0.718. `home_credit`, `lendingclub` and `german` are among our datasets, but under a different protocol.

In [ ]:
FIGS.save(literature_plots.credit_benchmark_landscape(), "literature_landscape",
    caption="Reported ROC-AUC on real credit datasets drawn from the tfm-library, one horizontal bar per published result, each on its paper's own full-dataset protocol rather than the in-context setting used elsewhere in this notebook.");

## Summary

The benchmark in text: what development selects (A), then how it does on the holdout (B). Printed
last, in the order of the sections above, so `output/All_Results.md` carries the same story as this
notebook, followed by the `tfm-library` sources it cites (pin `e5ce016`) and the list of figures.

In [ ]:
print(results_plots.results_summary(TASK, exp=EXP))
print()
print(literature_plots.summary())
print()
print(literature.references_md(["tanna_tabicl_hc", "tanna_tabpfn_hc", "creditg_tabpfn", "tanna_zerorecall"]))
print()
print(FIGS.summary())